# Task 2 — Extract PDF tiếng Việt, clean text và tách câu

Notebook này chỉ làm đúng phạm vi **Task 2**:

```text
PDF tiếng Việt → raw text → clean text → tách câu → xuất file cho Task 3
```

Notebook **không chạy OCR ảnh Hán** và **không chạy dóng hàng/alignment**.

Output chính cho Task 3 là:

```text
output_task2/<work_id>_viet_sentences.txt
output_task2/<work_id>_viet_sentences.jsonl
output_task2/<work_id>_viet_sentences.csv
output_task2/<work_id>_viet_sentences.xlsx
```

Bản `.txt` dùng để align nhanh. Bản `.jsonl/.csv/.xlsx` giữ thêm `page` và `sent_id` để debug khi dóng hàng.

## 1. Cài dependency nhẹ cho Task 2

Không cài PaddleOCR, không cài OCR GPU. Task 2 chỉ cần đọc PDF, clean text và tách câu tiếng Việt.

In [ ]:
# Chạy cell này nếu môi trường chưa có các package bên dưới.
# Trên Kaggle/Colab có internet thì lệnh này sẽ cài được.

%pip install -q pymupdf pypdf underthesea pandas openpyxl

## 2. Cấu hình đường dẫn

Nếu bạn dùng repo đã sửa kèm notebook này, đặt `REPO_DIR` trỏ vào thư mục repo.

Với Kaggle, thường repo nằm ở `/kaggle/working/...`. Với local, có thể để `Path.cwd()`.

`START_PAGE` rất quan trọng: nếu PDF có bìa, lời nói đầu, mục lục không tương ứng với bản Hán/Nôm thì nên tăng `START_PAGE` để bỏ phần đó.

In [ ]:
from pathlib import Path
import os

# ====== Chỉnh các biến này theo môi trường của bạn ======

# Nếu đang đứng trong repo, để Path.cwd() là đủ.
# Nếu repo ở Kaggle, có thể đổi thành Path("/kaggle/working/SinoNom-NLP-task2") hoặc Path("/kaggle/working/SinoNom-NLP")
REPO_DIR = Path.cwd()

# Nếu notebook chưa chạy trong repo, thử tự tìm repo phổ biến.
repo_candidates = [
    REPO_DIR,
    Path("/kaggle/working/SinoNom-NLP-task2"),
    Path("/kaggle/working/SinoNom-NLP-master"),
    Path("/kaggle/working/SinoNom-NLP"),
    Path("/mnt/data/SinoNom-NLP-task2"),
    Path("/mnt/data/task2_repo/SinoNom-NLP-master"),
]

for candidate in repo_candidates:
    if (candidate / "dataset" / "vietnam").exists() or (candidate / "main.py").exists():
        REPO_DIR = candidate
        break

PDF_PATH = REPO_DIR / "dataset" / "vietnam" / "q1.pdf"
OUTPUT_DIR = REPO_DIR / "output_task2"
WORK_ID = "q1"

# Dùng để bỏ bìa/lời nói đầu/mục lục nếu cần.
# Ban đầu để 1, sau khi preview nếu thấy chưa vào nội dung chính thì tăng lên.
START_PAGE = 1
END_PAGE = None  # ví dụ 50 nếu chỉ muốn test nhanh

print("REPO_DIR  =", REPO_DIR)
print("PDF_PATH  =", PDF_PATH)
print("OUTPUT_DIR=", OUTPUT_DIR)
print("PDF exists:", PDF_PATH.exists())

## 3. Thêm module riêng cho Task 2 vào repo

Cell này ghi file `nlp/vietnamese_pdf_processor.py`. Module này độc lập với OCR/alignment nên không bị kéo các dependency nặng như PaddleOCR hoặc Gemini.

In [ ]:
from pathlib import Path

(REPO_DIR / "nlp").mkdir(parents=True, exist_ok=True)
module_path = REPO_DIR / "nlp" / "vietnamese_pdf_processor.py"
module_path.write_text('"""Utilities for Task 2: Vietnamese PDF extraction, cleaning, and sentence splitting.\n\nThis module is intentionally independent from OCR and alignment components.\nIt can be used from a notebook, from scripts, or from the original pipeline.\n"""\n\nfrom __future__ import annotations\n\nimport csv\nimport json\nimport re\nimport unicodedata\nfrom collections import Counter\nfrom dataclasses import dataclass\nfrom pathlib import Path\nfrom typing import Any, Dict, Iterable, List, Optional, Sequence, Tuple\n\n\n@dataclass\nclass PDFPageText:\n    page: int\n    text: str\n\n\n@dataclass\nclass SentenceRecord:\n    work_id: str\n    lang: str\n    page: int\n    sent_id: int\n    text: str\n\n\ndef normalize_text(text: str) -> str:\n    """Normalize Unicode and common PDF punctuation artifacts."""\n    text = unicodedata.normalize("NFC", text or "")\n    replacements = {\n        "\\u00a0": " ",\n        "\\u200b": "",\n        "\\ufeff": "",\n        "“": \'"\',\n        "”": \'"\',\n        "‘": "\'",\n        "’": "\'",\n        "–": "-",\n        "—": "-",\n        "…": "...",\n    }\n    for old, new in replacements.items():\n        text = text.replace(old, new)\n    return text\n\n\ndef extract_pdf_pages(pdf_path: str | Path, start_page: int = 1, end_page: Optional[int] = None) -> List[PDFPageText]:\n    """Extract text page by page from a text-based PDF.\n\n    Page numbers are 1-based. The function first tries PyMuPDF because it is\n    usually more stable for layout-heavy PDFs, then falls back to pypdf/PyPDF2.\n    """\n    pdf_path = Path(pdf_path)\n    if not pdf_path.exists():\n        raise FileNotFoundError(f"PDF not found: {pdf_path}")\n\n    start_page = max(1, int(start_page or 1))\n    pages: List[PDFPageText] = []\n\n    try:\n        import fitz  # PyMuPDF\n\n        doc = fitz.open(str(pdf_path))\n        total_pages = len(doc)\n        last_page = total_pages if end_page is None else min(total_pages, int(end_page))\n        for page_no in range(start_page, last_page + 1):\n            page = doc.load_page(page_no - 1)\n            text = page.get_text("text") or ""\n            pages.append(PDFPageText(page=page_no, text=normalize_text(text)))\n        doc.close()\n        return pages\n    except ImportError:\n        pass\n\n    try:\n        try:\n            from pypdf import PdfReader\n        except ImportError:\n            from PyPDF2 import PdfReader\n\n        reader = PdfReader(str(pdf_path))\n        total_pages = len(reader.pages)\n        last_page = total_pages if end_page is None else min(total_pages, int(end_page))\n        for page_no in range(start_page, last_page + 1):\n            text = reader.pages[page_no - 1].extract_text() or ""\n            pages.append(PDFPageText(page=page_no, text=normalize_text(text)))\n        return pages\n    except ImportError as exc:\n        raise ImportError("Install pymupdf or pypdf to extract PDF text: pip install pymupdf pypdf") from exc\n\n\ndef _line_key(line: str) -> str:\n    line = normalize_text(line).strip().lower()\n    line = re.sub(r"\\s+", " ", line)\n    line = re.sub(r"\\d+", "#", line)\n    return line\n\n\ndef detect_repeated_lines(\n    pages: Sequence[PDFPageText],\n    threshold_ratio: float = 0.25,\n    min_occurrences: int = 3,\n    max_chars: int = 100,\n) -> set[str]:\n    """Detect likely headers/footers repeated across many pages."""\n    if not pages:\n        return set()\n\n    per_page_keys: List[set[str]] = []\n    for page in pages:\n        keys = set()\n        for raw_line in page.text.splitlines():\n            line = raw_line.strip()\n            if not line or len(line) > max_chars:\n                continue\n            key = _line_key(line)\n            if key:\n                keys.add(key)\n        per_page_keys.append(keys)\n\n    counts: Counter[str] = Counter()\n    for keys in per_page_keys:\n        counts.update(keys)\n\n    threshold = max(min_occurrences, int(len(pages) * threshold_ratio))\n    return {key for key, cnt in counts.items() if cnt >= threshold}\n\n\ndef _looks_like_page_number(line: str) -> bool:\n    line = line.strip()\n    if re.fullmatch(r"[-–—]?\\s*\\d{1,4}\\s*[-–—]?", line):\n        return True\n    if re.fullmatch(r"trang\\s+\\d{1,4}", line, flags=re.IGNORECASE):\n        return True\n    return False\n\n\ndef _looks_like_noise(line: str) -> bool:\n    line = line.strip()\n    if not line:\n        return True\n    if line.startswith("file://") or "file:///" in line:\n        return True\n    if re.search(r"https?://\\S+", line):\n        return True\n    if re.fullmatch(r"[\\W_]+", line, flags=re.UNICODE):\n        return True\n    return False\n\n\ndef clean_vietnamese_pages(\n    pages: Sequence[PDFPageText],\n    remove_repeated: bool = True,\n    repeated_threshold_ratio: float = 0.25,\n    min_line_chars: int = 2,\n    custom_drop_patterns: Optional[Sequence[str]] = None,\n) -> Tuple[List[PDFPageText], Dict[str, Any]]:\n    """Clean extracted Vietnamese PDF text while preserving page metadata."""\n    repeated_keys = detect_repeated_lines(pages, threshold_ratio=repeated_threshold_ratio) if remove_repeated else set()\n    compiled_custom = [re.compile(p, flags=re.IGNORECASE) for p in (custom_drop_patterns or [])]\n\n    cleaned_pages: List[PDFPageText] = []\n    stats = {\n        "input_pages": len(pages),\n        "removed_empty_or_noise_lines": 0,\n        "removed_page_number_lines": 0,\n        "removed_repeated_lines": 0,\n        "removed_custom_pattern_lines": 0,\n        "kept_lines": 0,\n        "repeated_keys": sorted(repeated_keys),\n    }\n\n    for page in pages:\n        kept: List[str] = []\n        for raw_line in normalize_text(page.text).splitlines():\n            line = re.sub(r"[ \\t]+", " ", raw_line).strip()\n            if len(line) < min_line_chars or _looks_like_noise(line):\n                stats["removed_empty_or_noise_lines"] += 1\n                continue\n            if _looks_like_page_number(line):\n                stats["removed_page_number_lines"] += 1\n                continue\n            if _line_key(line) in repeated_keys:\n                stats["removed_repeated_lines"] += 1\n                continue\n            if any(pat.search(line) for pat in compiled_custom):\n                stats["removed_custom_pattern_lines"] += 1\n                continue\n            kept.append(line)\n            stats["kept_lines"] += 1\n\n        # Join lines into paragraph-like text. Blank lines are intentionally not\n        # reconstructed because most PDF extractors do not preserve them well.\n        text = "\\n".join(kept)\n        text = re.sub(r"\\n{3,}", "\\n\\n", text).strip()\n        cleaned_pages.append(PDFPageText(page=page.page, text=text))\n\n    return cleaned_pages, stats\n\n\ndef pages_to_text(pages: Sequence[PDFPageText], include_page_markers: bool = True) -> str:\n    chunks: List[str] = []\n    for page in pages:\n        if include_page_markers:\n            chunks.append(f"<PAGE {page.page}>\\n{page.text}".strip())\n        else:\n            chunks.append(page.text.strip())\n    return "\\n\\n".join(chunk for chunk in chunks if chunk)\n\n\ndef _regex_sentence_split(text: str) -> List[str]:\n    """Vietnamese sentence splitter fallback.\n\n    Underthesea is preferred. This fallback avoids hard dependency on it so that\n    the notebook still runs in restricted environments.\n    """\n    text = re.sub(r"\\s+", " ", normalize_text(text)).strip()\n    if not text:\n        return []\n    # Split after sentence-final punctuation, but keep punctuation.\n    parts = re.split(r"(?<=[.!?;:])\\s+", text)\n    return [p.strip() for p in parts if p.strip()]\n\n\ndef get_vietnamese_sentence_tokenizer(prefer_underthesea: bool = True):\n    if prefer_underthesea:\n        try:\n            from underthesea import sent_tokenize\n\n            return sent_tokenize, "underthesea"\n        except Exception:\n            pass\n    return _regex_sentence_split, "regex"\n\n\ndef segment_vietnamese_pages(\n    pages: Sequence[PDFPageText],\n    work_id: str,\n    prefer_underthesea: bool = True,\n    min_sentence_chars: int = 2,\n) -> Tuple[List[SentenceRecord], str]:\n    tokenizer, tokenizer_name = get_vietnamese_sentence_tokenizer(prefer_underthesea=prefer_underthesea)\n    records: List[SentenceRecord] = []\n    sent_id = 1\n\n    for page in pages:\n        # Treat each non-empty line as a paragraph candidate. This helps avoid\n        # merging unrelated headings and body text into a single long segment.\n        paragraphs = [p.strip() for p in re.split(r"\\n+", page.text) if p.strip()]\n        for para in paragraphs:\n            normalized_para = re.sub(r"\\s+", " ", para).strip()\n            if not normalized_para:\n                continue\n            try:\n                sentences = tokenizer(normalized_para)\n            except Exception:\n                sentences = _regex_sentence_split(normalized_para)\n                tokenizer_name = "regex"\n\n            for sent in sentences:\n                sent = re.sub(r"\\s+", " ", normalize_text(sent)).strip()\n                if len(sent) < min_sentence_chars:\n                    continue\n                records.append(SentenceRecord(work_id=work_id, lang="viet", page=page.page, sent_id=sent_id, text=sent))\n                sent_id += 1\n\n    return records, tokenizer_name\n\n\ndef save_text(path: str | Path, text: str) -> None:\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    path.write_text(text, encoding="utf-8")\n\n\ndef save_sentence_txt(path: str | Path, records: Sequence[SentenceRecord]) -> None:\n    save_text(path, "\\n".join(record.text for record in records))\n\n\ndef save_sentence_jsonl(path: str | Path, records: Sequence[SentenceRecord]) -> None:\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    with path.open("w", encoding="utf-8") as f:\n        for record in records:\n            f.write(json.dumps(record.__dict__, ensure_ascii=False) + "\\n")\n\n\ndef save_sentence_csv(path: str | Path, records: Sequence[SentenceRecord]) -> None:\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    with path.open("w", encoding="utf-8", newline="") as f:\n        writer = csv.DictWriter(f, fieldnames=["work_id", "lang", "page", "sent_id", "text"])\n        writer.writeheader()\n        for record in records:\n            writer.writerow(record.__dict__)\n\n\ndef save_sentence_xlsx(path: str | Path, records: Sequence[SentenceRecord]) -> None:\n    try:\n        import pandas as pd\n    except ImportError as exc:\n        raise ImportError("Install pandas and openpyxl to export xlsx: pip install pandas openpyxl") from exc\n    df = pd.DataFrame([record.__dict__ for record in records])\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    df.to_excel(path, index=False)\n\n\ndef process_vietnamese_pdf(\n    pdf_path: str | Path,\n    output_dir: str | Path,\n    work_id: Optional[str] = None,\n    start_page: int = 1,\n    end_page: Optional[int] = None,\n    prefer_underthesea: bool = True,\n    custom_drop_patterns: Optional[Sequence[str]] = None,\n) -> Dict[str, Any]:\n    """Run the full Task 2 flow for one Vietnamese PDF."""\n    pdf_path = Path(pdf_path)\n    output_dir = Path(output_dir)\n    work_id = work_id or pdf_path.stem\n\n    raw_pages = extract_pdf_pages(pdf_path, start_page=start_page, end_page=end_page)\n    cleaned_pages, clean_stats = clean_vietnamese_pages(raw_pages, custom_drop_patterns=custom_drop_patterns)\n    sentences, tokenizer_name = segment_vietnamese_pages(cleaned_pages, work_id=work_id, prefer_underthesea=prefer_underthesea)\n\n    raw_text = pages_to_text(raw_pages, include_page_markers=True)\n    clean_text = pages_to_text(cleaned_pages, include_page_markers=True)\n\n    raw_txt = output_dir / f"{work_id}_viet_raw.txt"\n    clean_txt = output_dir / f"{work_id}_viet_clean.txt"\n    sent_txt = output_dir / f"{work_id}_viet_sentences.txt"\n    sent_jsonl = output_dir / f"{work_id}_viet_sentences.jsonl"\n    sent_csv = output_dir / f"{work_id}_viet_sentences.csv"\n    sent_xlsx = output_dir / f"{work_id}_viet_sentences.xlsx"\n    stats_json = output_dir / f"{work_id}_viet_task2_stats.json"\n\n    save_text(raw_txt, raw_text)\n    save_text(clean_txt, clean_text)\n    save_sentence_txt(sent_txt, sentences)\n    save_sentence_jsonl(sent_jsonl, sentences)\n    save_sentence_csv(sent_csv, sentences)\n    try:\n        save_sentence_xlsx(sent_xlsx, sentences)\n        xlsx_path = str(sent_xlsx)\n    except Exception:\n        xlsx_path = None\n\n    stats = {\n        "work_id": work_id,\n        "pdf_path": str(pdf_path),\n        "start_page": start_page,\n        "end_page": end_page,\n        "raw_pages": len(raw_pages),\n        "clean_stats": clean_stats,\n        "tokenizer": tokenizer_name,\n        "num_sentences": len(sentences),\n        "outputs": {\n            "raw_txt": str(raw_txt),\n            "clean_txt": str(clean_txt),\n            "sentences_txt": str(sent_txt),\n            "sentences_jsonl": str(sent_jsonl),\n            "sentences_csv": str(sent_csv),\n            "sentences_xlsx": xlsx_path,\n            "stats_json": str(stats_json),\n        },\n    }\n    save_text(stats_json, json.dumps(stats, ensure_ascii=False, indent=2))\n    return stats\n', encoding="utf-8")
print("Wrote:", module_path)

## 4. Preview vài trang PDF để chọn `START_PAGE`

Chạy cell này để xem PDF extract ra gì ở các trang đầu. Nếu thấy vẫn là bìa/lời nói đầu/mục lục, hãy quay lại cell cấu hình và tăng `START_PAGE`.

Ví dụ: `START_PAGE = 10` hoặc `START_PAGE = 15`.

In [ ]:
import sys
from IPython.display import display, Markdown

sys.path.insert(0, str(REPO_DIR))
from nlp.vietnamese_pdf_processor import extract_pdf_pages

preview_pages = extract_pdf_pages(PDF_PATH, start_page=START_PAGE, end_page=min((END_PAGE or START_PAGE + 2), START_PAGE + 2))

for page in preview_pages:
    text = page.text.strip().replace("
", " ")
    text = text[:1200] + ("..." if len(text) > 1200 else "")
    display(Markdown(f"### Page {page.page}

```text
{text}
```"))

## 5. Chạy Task 2

Cell này sẽ tạo các file:

```text
<work_id>_viet_raw.txt
<work_id>_viet_clean.txt
<work_id>_viet_sentences.txt
<work_id>_viet_sentences.jsonl
<work_id>_viet_sentences.csv
<work_id>_viet_sentences.xlsx
<work_id>_viet_task2_stats.json
```

In [ ]:
from nlp.vietnamese_pdf_processor import process_vietnamese_pdf

stats = process_vietnamese_pdf(
    pdf_path=PDF_PATH,
    output_dir=OUTPUT_DIR,
    work_id=WORK_ID,
    start_page=START_PAGE,
    end_page=END_PAGE,
    prefer_underthesea=True,
    # Có thể thêm regex để bỏ dòng nhiễu nếu thấy lặp lại trong preview.
    # Ví dụ: custom_drop_patterns=[r"^QUỐC SỬ QUAN", r"^VIỆN SỬ HỌC"]
    custom_drop_patterns=[],
)

stats

## 6. Kiểm tra nhanh kết quả

Mục tiêu là xem câu đầu/cuối có còn rác như bìa, số trang, header/footer hay không.

In [ ]:
import json
from pathlib import Path

sent_path = Path(stats["outputs"]["sentences_txt"])
clean_path = Path(stats["outputs"]["clean_txt"])
stats_path = Path(stats["outputs"]["stats_json"])

sentences = [line.strip() for line in sent_path.read_text(encoding="utf-8").splitlines() if line.strip()]

print("Số câu:", len(sentences))
print("Clean text:", clean_path)
print("Sentences:", sent_path)
print("Stats:", stats_path)

print("
--- 20 câu đầu ---")
for i, sent in enumerate(sentences[:20], start=1):
    print(f"{i:03d}. {sent}")

print("
--- 10 câu cuối ---")
for i, sent in enumerate(sentences[-10:], start=max(1, len(sentences)-9)):
    print(f"{i:03d}. {sent}")

## 7. Xem bảng metadata cho Task 3

Bảng này giúp người làm alignment biết mỗi câu đến từ trang nào.

In [ ]:
import pandas as pd

csv_path = Path(stats["outputs"]["sentences_csv"])
df = pd.read_csv(csv_path)
display(df.head(20))

print(df.describe(include="all"))

## 8. Đóng gói output Task 2

File zip này có thể gửi thẳng cho người làm Task 3.

In [ ]:
import zipfile

zip_path = OUTPUT_DIR / f"{WORK_ID}_task2_outputs.zip"
files_to_zip = [
    Path(stats["outputs"]["raw_txt"]),
    Path(stats["outputs"]["clean_txt"]),
    Path(stats["outputs"]["sentences_txt"]),
    Path(stats["outputs"]["sentences_jsonl"]),
    Path(stats["outputs"]["sentences_csv"]),
    Path(stats["outputs"]["stats_json"]),
]
if stats["outputs"].get("sentences_xlsx"):
    files_to_zip.append(Path(stats["outputs"]["sentences_xlsx"]))

with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for path in files_to_zip:
        if path.exists():
            zf.write(path, arcname=path.name)

print("Created:", zip_path)
print("Files:")
for path in files_to_zip:
    print("-", path.name, "OK" if path.exists() else "MISSING")

## 9. Ghi chú bàn giao

Khi bàn giao cho Task 3, nên dùng:

```text
<work_id>_viet_sentences.txt
```

Nếu align bị lệch, dùng file metadata để debug:

```text
<work_id>_viet_sentences.jsonl
<work_id>_viet_sentences.xlsx
```

Nếu 20 câu đầu vẫn là bìa/lời nói đầu/mục lục, đừng sửa alignment vội. Hãy quay lại chỉnh `START_PAGE` rồi chạy lại Task 2.